# fintrack showcase

This notebook walks through the package the way you would use it from Python
rather than from the command line: load a CSV, categorise it, find the
recurring payments, project the balance and read the recommendations.


In [ ]:
import fintrack

fintrack.__version__


## 1. Load the bundled example data

`load_transactions` figures out the delimiter, the column names, the date
format and the decimal separator by itself.


In [ ]:
transactions = fintrack.load_transactions(fintrack.sample_data_path())
print(f'{len(transactions)} transactions from '
      f'{transactions[0].date} to {transactions[-1].date}')
for item in transactions[:5]:
    print(item)


## 2. Categorise

The categorizer is keyword based. Longer keywords win, so a rule for
`amazon prime` beats the generic `amazon` rule.


In [ ]:
categorizer = fintrack.Categorizer()
categorizer.add_rule('Hobbies', 'Thomann', 'Kletterhalle')
transactions = categorizer.categorize_all(transactions)

spending = fintrack.spending_by_category(transactions)
for name, amount in list(spending.items())[:8]:
    print(f'{name:<22} {amount:>10.2f}')


## 3. Monthly aggregation


In [ ]:
summaries = fintrack.monthly_summaries(transactions)
for summary in summaries[-6:]:
    print(f'{summary.month}  income {summary.income:>9.2f}  '
          f'expenses {summary.expenses:>9.2f}  saved {summary.savings_rate:.0%}')


## 4. Detect recurring payments

Each series carries the detected period, a confidence score, the next
expected date, and its cost normalised to a month and to a year.


In [ ]:
series = fintrack.detect_recurring(transactions)

for item in series:
    print(f'{item.label[:30]:<32} {item.period_label:<10} '
          f'{item.average_amount:>9.2f}  per year {item.yearly_equivalent:>10.2f}  '
          f'{item.confidence:.0%}')


Tightening the thresholds gives a shorter, more conservative list:


In [ ]:
strict = fintrack.detect_recurring(
    transactions, min_occurrences=6, amount_tolerance=0.05, min_confidence=0.9
)
print(f'{len(series)} series by default, {len(strict)} with strict settings')


## 5. Forecast the balance


In [ ]:
forecast = fintrack.forecast_balance(
    transactions, horizon_days=120, starting_balance='2400.00', series=series
)

print('start        ', forecast.starting_balance)
print('end          ', forecast.ending_balance)
print('daily extra  ', forecast.daily_discretionary)
print('lowest point ', forecast.minimum_point.balance, 'on',
      forecast.minimum_point.date)
print('goes negative', forecast.first_negative)


What happens if we start from a much thinner balance?


In [ ]:
tight = fintrack.forecast_balance(
    transactions, horizon_days=120, starting_balance='150.00', series=series
)
print(tight.first_negative)


## 6. Recommendations


In [ ]:
for advice in fintrack.generate_recommendations(summaries, series, forecast):
    print(advice)
    print()


## 7. Charts

The plotting functions never display anything, they always write a PNG and
return the path. That keeps them usable on headless machines.


In [ ]:
from fintrack.visualization import plot_all

paths = plot_all(summaries, spending, series, forecast, 'output',
                 history=transactions)
paths


In [ ]:
from IPython.display import Image

Image(filename='output/balance_forecast.png')


## 8. The whole report in one call


In [ ]:
report = fintrack.build_report(
    transactions, summaries, spending, series, forecast,
    fintrack.generate_recommendations(summaries, series, forecast),
    chart_paths=paths,
)
print(report)
